# 03 — Model experiments

Train **Logistic Regression**, **Random Forest**, and **Gradient Boosting** on the same preprocessor; select the best model by **ROC-AUC** on the held-out test set (identical logic to [`training_pipeline`](../src/pipeline/training_pipeline.py)).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd().parent))
from src.data.load_data import load_raw_data
from src.features.build_features import build_features
from src.data.preprocess import prepare_target, build_preprocessor
from src.models.compare_models import compare_models
%matplotlib inline

In [ ]:
RAW = Path("../data/raw/telco_customer_churn.csv")
df = load_raw_data(RAW)
X, y = prepare_target(build_features(df))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
preprocessor = build_preprocessor(random_state=42)
print(f"Train {len(X_train):,} | Test {len(X_test):,} | Pos rate train: {y_train.mean():.3f}")

In [ ]:
best_name, comparison_df, best_metrics, best_pipeline = compare_models(
    preprocessor, X_train, y_train, X_test, y_test, random_state=42
)
print("=" * 50)
print("SELECTED MODEL:", best_name)
print("=" * 50)
print("Test-set metrics (default 0.5 threshold for precision/recall/f1/acc):")
for k in ["roc_auc", "precision", "recall", "f1", "accuracy"]:
    print(f"  {k:12s} {best_metrics.get(k, 0):.4f}")
print("\nFull comparison:")
print(comparison_df.to_string(index=False))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(comparison_df["model"], comparison_df["roc_auc"], color="steelblue", edgecolor="black")
ax.set_xlabel("ROC-AUC (test)")
ax.set_title("Model comparison")
ax.set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

## Interpretation

- **ROC-AUC** is the selection metric (threshold-free; robust under imbalance).
- **Precision / recall / F1** in the table use the classifier’s default 0.5 cutoff — the **production pipeline** picks a separate F1-optimal threshold on the test set (see `04_error_analysis.ipynb` and `reports/metrics/threshold_summary.json`).
- If **Gradient Boosting** wins but deployment size matters (e.g. Vercel), consider exporting a smaller model from the registry in a separate experiment.